# 红利低波数据下载

本 notebook 只负责数据更新。参数来自 `config.yaml`；`active_source` 保证每次运行只访问一个外部数据源。默认运行 RQData 的成份股、PB 和分钟行情下载。

In [1]:
import os
import sys
from pathlib import Path

project_root = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").exists()
)
os.chdir(project_root)
source_root = project_root / "src"
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

In [2]:
from importlib.resources import files

import pandas as pd
import yaml

from pyquant import (
    get_period_end_dates,
    load_dataset,
    update_dataset,
    update_minute_data,
)
from strategies.div_low_vol.components import (
    build_intraday_minute_requests,
    select_div_low_vol_candidates,
)
from strategies.div_low_vol.components import (
    select_div_low_vol_download_symbols,
)

with (
    files("strategies.div_low_vol")
    .joinpath("config.yaml")
    .open(encoding="utf-8") as stream
):
    config = yaml.safe_load(stream) or {}
active_source = "rqdata"
if active_source not in {"rqdata", "baostock", "csindex"}:
    raise ValueError(f"Unsupported data source: {active_source}")
start_date = pd.Timestamp(config["data"]["start_date"])
end_date = pd.Timestamp(config["data"]["end_date"])
pool = config["data"]["pool"]
lookback_date = start_date - pd.DateOffset(years=3)
dividend_start = (
    lookback_date - pd.DateOffset(years=1)
    if (start_date.month == 12 and start_date.day <= 20)
    else lookback_date
)

## 官方中证指数

仅当 `active_source == "csindex"` 时运行；不会初始化 BaoStock 或 RQData。

In [ ]:
if active_source == "csindex":
    official_index_job = update_dataset(
        "csindex_daily",
        start=start_date.strftime("%Y-%m-%d"),
        end=end_date.strftime("%Y-%m-%d"),
        pool=["H30269", "H20269"],
    )
    official_index_job.wait()

## 官方历史成份股

仅当 `active_source == "rqdata"` 时运行，只调用 RQData 和本地 DuckDB。

In [ ]:
if active_source == "rqdata":
    constituent_job = update_dataset(
        "index_constituents",
        start=start_date.strftime("%Y-%m-%d"),
        end=end_date.strftime("%Y-%m-%d"),
        pool=[config["strategy_3"]["index_code"]],
    )
    constituent_snapshots = constituent_job.wait()

## RQData 六口径 PB

仅当 `active_source == "rqdata"` 时运行。股票池、PB 和交易日信息全部来自 RQData，不读取或更新其他数据源。

In [ ]:
if active_source == "rqdata":
    valuation_start = start_date.to_period("M").start_time
    pb_job = update_dataset(
        "stock_pb_daily",
        start=valuation_start.strftime("%Y-%m-%d"),
        end=end_date.strftime("%Y-%m-%d"),
        pool="all",
    )
    pb_downloads = pb_job.wait()

## 日行情

仅当 `active_source == "baostock"` 时运行。先等待完成，再生成分红和股本下载股票池。

In [ ]:
if active_source == "baostock":
    download_job = update_dataset(
        "stock_daily",
        start=lookback_date.strftime("%Y-%m-%d"),
        end=end_date.strftime("%Y-%m-%d"),
        pool=pool,
    )

### 下载控制

In [ ]:
if active_source == "baostock":
    download_job.pause()
    download_job.state

In [ ]:
if active_source == "baostock":
    download_job.resume()
    download_job.state

In [ ]:
if active_source == "baostock":
    download_job.stop()
    download_job.wait()
    download_job.state

## 分红与股本下载股票池

仅当 `active_source == "baostock"` 时运行。按完整下载区间内至少 720 条有效行情筛选；结果覆盖 `pool`，供分红和股本共用。

In [ ]:
if active_source == "baostock":
    price = load_dataset(
        "stock_daily",
        start=lookback_date.strftime("%Y-%m-%d"),
        end=end_date.strftime("%Y-%m-%d"),
    )
    adjust_factor_job = update_dataset(
        "stock_adjust_factor",
        start="1990-01-01",
        end=end_date.strftime("%Y-%m-%d"),
        pool=config["data"]["pool"],
    )
    adjust_factor_downloads = adjust_factor_job.wait()
    pool = select_div_low_vol_download_symbols(price, end_date, config)
    if not pool:
        raise ValueError("No symbols have at least 720 valid prices in the download range")
    print(f"Dividend/share download pool: {len(pool)} symbols")

## 分红数据

In [ ]:
if active_source == "baostock":
    download_job = update_dataset(
        "dividend",
        start=dividend_start.strftime("%Y-%m-%d"),
        end=end_date.strftime("%Y-%m-%d"),
        pool=pool,
    )

## 季度总股本

In [ ]:
if active_source == "baostock":
    download_job = update_dataset(
        "stock_profit_quarterly",
        start=lookback_date.strftime("%Y-%m-%d"),
        end=end_date.strftime("%Y-%m-%d"),
        pool=pool,
    )

## 1 分钟行情

仅当 `active_source == "rqdata"` 时运行。请求生成只读取本地 DuckDB，不会自动下载日行情、分红或季度总股本，也不会访问其他外部数据源。

In [3]:
if active_source == "rqdata":
    price = load_dataset(
        "stock_daily",
        start=lookback_date.strftime("%Y-%m-%d"),
        end=end_date.strftime("%Y-%m-%d"),
    )
    dividends = load_dataset("dividend")
    dividend_queries = load_dataset("dividend_queries")
    shares = load_dataset("stock_profit_quarterly")
    trading_dates = price["date"].drop_duplicates().sort_values()
    rebalance_dates = get_period_end_dates(
        trading_dates[trading_dates.between(start_date, end_date)]
    )
    candidate_config = {
        "universe": config["universe"],
        "selection": config["strategy_2"],
    }
    minute_config = config["minute_data"]
    minute_requests = []
    for signal_date in rebalance_dates:
        candidates = select_div_low_vol_candidates(
            price, dividends, dividend_queries, shares, signal_date, candidate_config
        )
        minute_requests.extend(
            build_intraday_minute_requests(
                candidates.index.tolist(),
                signal_date,
                trading_dates,
                lookback_trading_days=candidate_config["selection"]["lookback_trading_days"],
                max_candidates=candidate_config["selection"]["dividend_top_n"],
            )
        )
    if not minute_requests:
        raise ValueError("No minute-data requests were generated")
    print(f"Minute-data requests: {len(minute_requests)}")

Minute-data requests: 17100


In [4]:
if active_source == "rqdata":
    minute_job = update_minute_data(
        minute_requests,
        max_attempts=minute_config["max_attempts"],
        quota_reserve_bytes=minute_config["quota_reserve_bytes"],
        min_bars_per_day=minute_config["min_bars_per_day"],
    )
    minute_downloads = minute_job.wait()
    minute_downloads["status"].value_counts(dropna=False)

Updated 1395/1395

Minute-data update completed: 1395/1395 tasks processed.
